<a href="https://colab.research.google.com/github/shan369rpa/IA-MEDIA/blob/feature%2Fcolab-pipeline/demo_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Ô 1: SETUP ENVIRONMENT
# ---------------------------------
# @title ⬅️ Step 0: Click here to Set Up the Environment
# @markdown This cell will clone the project repository and install all necessary libraries from `requirements.txt`.

import os
import sys

print("🚀 Starting environment setup...")

# 1. Clone the project repository from GitHub
print("Cloning the IA-MEDIA repository...")
if os.path.exists('IA-MEDIA'):
    !rm -rf IA-MEDIA
# !!! THAY THẾ 'YourUsername' và 'dev' bằng tên user/nhánh GitHub của bạn !!!
!git clone --branch dev https://github.com/shan369rpa/IA-MEDIA.git

# 2. Install all dependencies from the requirements.txt file
print("\nInstalling required Python libraries...")
!pip install -q -r IA-MEDIA/requirements.txt

# 3. Add the project's source code to the Python path
if '/content/IA-MEDIA' not in sys.path:
    sys.path.append('/content/IA-MEDIA')

print("\n✅ Environment is ready! You can now proceed to the next steps.")

🚀 Starting environment setup...
Cloning the IA-MEDIA repository...
Cloning into 'IA-MEDIA'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 247 (delta 103), reused 180 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (247/247), 1.97 MiB | 22.40 MiB/s, done.
Resolving deltas: 100% (103/103), done.

Installing required Python libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━

In [10]:
# Ô 2: MOUNT DRIVE & LOAD CONFIG
# ---------------------------------
# @title ⬅️ Step 1: Connect Google Drive & Load Secrets
# @markdown This cell connects to your Google Drive to access source data and load secret configurations.

from google.colab import drive
from dotenv import load_dotenv
import os

print("🔌 Attempting to connect to Google Drive...")
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive connected successfully.")
except Exception as e:
    print(f"❌ ERROR: Failed to mount Google Drive: {e}")
    raise e

# Path to the secret .env file on your Google Drive
env_path = '/content/drive/MyDrive/IA_MEDIA_PROJECT/secrets/.env'

if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path)
    print("✅ Secret environment variables loaded successfully.")
    # Quick check for a key variable
    if os.getenv("DB_HOST"):
        print(f"   - DB_HOST found: {os.getenv('DB_HOST')}")
    else:
        print("   - ⚠️ WARNING: DB_HOST not found. Database connection will likely fail.")
else:
    error_message = f"❌ ERROR: Secret file not found at '{env_path}'. Please ensure it exists and the path is correct."
    print(error_message)
    raise FileNotFoundError(error_message)

🔌 Attempting to connect to Google Drive...
Mounted at /content/drive
✅ Google Drive connected successfully.
✅ Secret environment variables loaded successfully.
   - DB_HOST found: localhost


In [11]:
# Ô 2.5: ESTABLISH SECURE SSH TUNNEL
# ---------------------------------
# @title ⬅️ Step 1.5: Connect to Database Server via SSH Tunnel
# @markdown This cell retrieves the SSH key from Google Drive and establishes a secure tunnel to your private database server.

import os
import time
import socket
import subprocess

print("🔗 Setting up Secure SSH Tunnel...")

# 1. Cấu hình đường dẫn
# Đường dẫn key trên Drive (phải khớp với tên bạn vừa đặt)
key_bath_drive = "/content/drive/MyDrive/IA_MEDIA_PROJECT/secrets/id_rsa_colab"
# Đường dẫn key trên máy ảo Colab
key_path_local = "/root/.ssh/id_rsa"
# IP Server của bạn
server_ip = "180.93.137.58"
# Cổng DB trên Server (Container mới)
remote_db_port = 5433
# Cổng Local trên Colab (khớp với file .env)
local_forward_port = 6000

# 2. Chuẩn bị thư mục SSH
if not os.path.exists("/root/.ssh"):
    os.makedirs("/root/.ssh")

# 3. Copy Key từ Drive và phân quyền
if os.path.exists(key_bath_drive):
    # Copy file
    !cp "{key_bath_drive}" "{key_path_local}"
    # Set quyền 600 (chỉ chủ sở hữu đọc/ghi) - Bắt buộc cho SSH
    !chmod 600 "{key_path_local}"
    print("✅ SSH Key loaded from Drive.")
else:
    print(f"❌ ERROR: SSH Key not found at: {key_bath_drive}")
    print("   Please upload 'colab_key' to that location and rename it to 'id_rsa_colab'.")
    # Dừng lại nếu không có key
    raise FileNotFoundError("SSH Key missing")

# 4. Thêm Server vào known_hosts (để tránh hỏi Yes/No)
print(f"   Adding {server_ip} to known_hosts...")
!ssh-keyscan -H {server_ip} >> /root/.ssh/known_hosts 2>/dev/null

# 5. Khởi tạo đường hầm (Chạy ngầm)
print(f"🚀 Establishing tunnel to {server_ip}:{remote_db_port}...")

# Kill tunnel cũ nếu có để tránh lỗi "Address already in use"
!pkill -f "ssh -f -N -L {local_forward_port}"

# Lệnh SSH:
# -f: Background mode
# -N: Do not execute remote command (forwarding only)
# -L: Port forwarding config
# -i: Identity file (Key)
cmd = f"ssh -f -N -L {local_forward_port}:localhost:{remote_db_port} root@{server_ip} -i {key_path_local}"
os.system(cmd)

# Chờ 2 giây để kết nối thiết lập
time.sleep(2)

# 6. Kiểm tra kết nối
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
result = sock.connect_ex(('127.0.0.1', local_forward_port))
sock.close()

if result == 0:
   print(f"✅ TUNNEL ESTABLISHED! Localhost:{local_forward_port} is now mapped to Server DB.")
else:
   print("❌ TUNNEL FAILED. Could not connect to local port.")
   print("   Check your Server IP, SSH Key, or Firewall settings.")

🔗 Setting up Secure SSH Tunnel...
✅ SSH Key loaded from Drive.
   Adding 180.93.137.58 to known_hosts...
🚀 Establishing tunnel to 180.93.137.58:5433...
✅ TUNNEL ESTABLISHED! Localhost:6000 is now mapped to Server DB.


In [12]:
# Ô 3: CONFIGURE BATCH PARAMETERS
# ---------------------------------
# @title ⬅️ Step 2: Configure Your Processing Batch
# @markdown ---
# @markdown ### ✏️ Please enter the name of the batch you want to process.
# @markdown (This must match the folder name in your Google Drive `source_data` directory)

BATCH_NAME = "Batch_01" #@param {type:"string"}

# --- The rest of the code runs automatically based on your input ---
SOURCE_DATA_DIR = f"/content/drive/MyDrive/IA_MEDIA_PROJECT/source_data/{BATCH_NAME}"
PROCESSED_DATA_DIR = f"/content/drive/MyDrive/IA_MEDIA_PROJECT/processed_data"
WORKSPACE_DIR = "/content/workspace" # Use Colab's fast local storage for processing

print(f"✅ Configuration set!")
print(f"📂 Source Directory: {SOURCE_DATA_DIR}")
print(f"📦 Results Directory: {PROCESSED_DATA_DIR}")
print(f"🏭 Workspace Directory: {WORKSPACE_DIR}")

✅ Configuration set!
📂 Source Directory: /content/drive/MyDrive/IA_MEDIA_PROJECT/source_data/Batch_01
📦 Results Directory: /content/drive/MyDrive/IA_MEDIA_PROJECT/processed_data
🏭 Workspace Directory: /content/workspace


In [13]:
# Ô 4: RUN CORE ANALYSIS PIPELINE
# ---------------------------------
# @title ⬅️ Step 3: Run Core Analysis Pipeline (Video -> Chunks)
# @markdown This cell executes the main processing pipeline. It will find all videos in your source directory, run analysis, and create thousands of small audio/video chunk files in the Colab temporary storage.
# @markdown ---
# @markdown **⚠️ This is the most time-consuming step. Please be patient.**

# Import necessary functions AFTER cloning the repo
from main import main_pipeline
import os
import logging

# Set up logging to show info in the cell output
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)

print("🏭 Starting the main analysis pipeline...")
print(f"This will process all videos found in '{SOURCE_DATA_DIR}'.")

# Ensure the workspace directory is clean for a new run
if os.path.exists(WORKSPACE_DIR):
    print("   - Removing old workspace directory to ensure a clean run...")
    !rm -rf {WORKSPACE_DIR}
os.makedirs(WORKSPACE_DIR, exist_ok=True)

# Run the main pipeline function, passing the necessary paths
try:
    # We will need to update main_pipeline to accept workspace_dir as an argument
    main_pipeline(source_dir=SOURCE_DATA_DIR, workspace_dir=WORKSPACE_DIR)
    print("\n✅ Core analysis pipeline completed successfully!")
    print(f"You can check the generated files in the file browser on the left, under `{WORKSPACE_DIR}`")
except Exception as e:
    logging.exception("An error occurred during the analysis pipeline.")
    print(f"\n❌ ERROR: The analysis pipeline failed. Please check the logs above.")

🏭 Starting the main analysis pipeline...
This will process all videos found in '/content/drive/MyDrive/IA_MEDIA_PROJECT/source_data/Batch_01'.
   - Removing old workspace directory to ensure a clean run...


2025-11-23 11:48:04,796 - INFO - ===== STARTING CORE ANALYSIS PIPELINE ON DIRECTORY: /content/drive/MyDrive/IA_MEDIA_PROJECT/source_data/Batch_01 =====
2025-11-23 11:48:04,801 - ERROR - No .fcpxml files found in '/content/drive/MyDrive/IA_MEDIA_PROJECT/source_data/Batch_01'. Nothing to process. Stopping.



✅ Core analysis pipeline completed successfully!
You can check the generated files in the file browser on the left, under `/content/workspace`


In [ ]:
# Ô 5: RUN VECTORIZATION PIPELINE
# ---------------------------------
# @title ⬅️ Step 4: Run Vectorization Pipeline (Chunks -> Database)
# @markdown This cell will scan all the generated audio chunks, create a vector embedding for each one, and insert the data into your remote PostgreSQL database.

from vectorize import main_vectorize
import os
import logging

print("🧠 Starting the vectorization pipeline...")

# Check if the output from the previous step exists
dataset_version = "tnh_speech_v0.1" # This should be a shared config
chunk_dir = os.path.join(WORKSPACE_DIR, "datasets", dataset_version)

if not os.path.exists(chunk_dir):
    print(f"❌ ERROR: Chunks directory not found at '{chunk_dir}'. Did the previous step (Core Analysis) fail or produce no output?")
else:
    try:
        # We will need to update main_vectorize to accept chunk_dir as an argument
        main_vectorize(chunk_dir=chunk_dir)
        print("\n✅ Vectorization pipeline completed successfully!")
        print("Data has been inserted into the database.")
    except Exception as e:
        logging.exception("An error occurred during the vectorization pipeline.")
        print(f"\n❌ ERROR: The vectorization pipeline failed. Please check the logs above.")

In [ ]:
# Ô 6: COMPRESS & BACKUP RESULTS
# ---------------------------------
# @title ⬅️ Step 5: Compress & Backup Results to Google Drive
# @markdown This final cell will compress the entire `workspace` directory (containing all generated chunks) into a single .zip file and save it to your Google Drive for permanent storage.

import os

# Define paths
output_zip_path = os.path.join(PROCESSED_DATA_DIR, f"{BATCH_NAME}_results.zip")

if not os.path.exists(WORKSPACE_DIR):
    print(f"❌ ERROR: Workspace directory '{WORKSPACE_DIR}' not found. Cannot create backup.")
else:
    print(f"🗜️ Compressing '{WORKSPACE_DIR}' into a zip file...")

    # Use a direct shell command for zipping, it's often faster
    # -r for recursive, -q for quiet to reduce log spam
    !zip -r -q {output_zip_path} {WORKSPACE_DIR}

    if os.path.exists(output_zip_path):
        # Get file size for user confirmation
        file_size_gb = os.path.getsize(output_zip_path) / (1024**3)
        print(f"✅ Successfully created backup file at:")
        print(f"   '{output_zip_path}'")
        print(f"   File size: {file_size_gb:.2f} GB")
    else:
        print(f"❌ ERROR: Failed to create zip file.")```